In [ ]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import *

# --- 1. CONFIGURATION ---
# "Files" points to the root folder of your Lakehouse
raw_path = "Files" 

# exact filenames -> target table names
files_map = {
    "olist_customers_dataset.csv": "bronze_customers",
    "olist_geolocation_dataset.csv": "bronze_geolocation",
    "olist_order_items_dataset.csv": "bronze_order_items",
    "olist_order_payments_dataset.csv": "bronze_order_payments",
    "olist_order_reviews_dataset.csv": "bronze_order_reviews",
    "olist_orders_dataset.csv": "bronze_orders",
    "olist_products_dataset.csv": "bronze_products",
    "olist_sellers_dataset.csv": "bronze_sellers",
    "product_category_name_translation.csv": "bronze_category_translation"
}

# --- 2. INGESTION FUNCTION ---
def ingest_to_bronze(file_name, table_name):
    print(f"⏳ Processing {file_name}...")
    try:
        # Construct the full path
        full_path = f"{raw_path}/{file_name}"
        
        # READ: Load CSV
        # inferSchema=True is okay for Bronze (raw) layer
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(full_path)
        
        # TRANSFORM: Add Audit Columns (Best Practice)
        df_enriched = df.withColumn("ingestion_timestamp", current_timestamp()) \
                        .withColumn("source_file", lit(file_name))
        
        # WRITE: Save as Delta Table
        # mode("overwrite") allows you to re-run this safely without errors
        df_enriched.write.format("delta").mode("overwrite").saveAsTable(table_name)
        
        print(f"✅ Success: {file_name} -> Tables/{table_name}")
        
    except Exception as e:
        print(f"❌ FAILED {file_name}: {e}")

# --- 3. EXECUTE ---
print(f"🚀 Starting ingestion of {len(files_map)} files from Root Folder...")
for csv, table in files_map.items():
    ingest_to_bronze(csv, table)


StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] This spark job can't be run because you have hit a spark compute or API rate limit. To run this spark job, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. HTTP status code: 430 {Learn more} HTTP status code: 430.